In [0]:
select * from parquet.`/Volumes/lakehouse_dev/datasphere_test/managedvolume/house-price.parquet`

In [0]:
use catalog lakehouse_dev;
use schema datasphere_test;
create or replace table house_price using delta as select * from parquet.`/Volumes/lakehouse_dev/datasphere_test/managedvolume/house-price.parquet`;

select * from house_price;

describe extended house_price;

describe schema datasphere_test;


create table house_price_bronze; 
copy into house_price_bronze
from ( select *, _metadata.file_path as file_path
from '/Volumes/lakehouse_dev/datasphere_test/managedvolume/house-price.parquet')
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');
select * from house_price_bronze;


insert overwrite table house_price_bronze
select *, _metadata.file_path as file_path from parquet.`/Volumes/lakehouse_dev/datasphere_test/managedvolume/house-price.parquet`;


describe history house_price_bronze;

In [0]:
--UnPivot

use catalog lakehouse_dev;
use schema datasphere_test;

create table if not exists sales (
  location STRING,
  year INT,
  q1 INT,
  q2 INT,
  q3 INT,
  q4 INT
);
insert into sales (location, year, q1, q2, q3, q4) values
('west', 2016, 1000, 1200, 1400, 1600),
('east', 2017, 1100, 1300, 1500, 1700),
('north', 2016, 1200, 1400, 1600, 1800),
('south', 2017, 1300, 1500, 1700, 1900);

select * from sales;

create or replace view sales_unpivot as
select * from sales unpivot exclude nulls 
(sales_value for quarter in (q1 as `Jan-Mar`,
 q2 as `Apr-Jun`,
 q3 as `Jul-Sep`,
 q4 as `Oct-Dec`));

 select * from sales_unpivot;



In [0]:
CREATE OR REPLACE TEMPORARY VIEW oncall (
  year, week, area, name1, email1, phone1, name2, email2, phone2
) AS
VALUES
  (
    2022,
    1,
    'frontend',
    'Freddy',
    'fred@alwaysup.org',
    15551234567,
    'Fanny',
    'fanny@lwaysup.org',
    15552345678
  ),
  (
    2022,
    1,
    'backend',
    'Boris',
    'boris@alwaysup.org',
    15553456789,
    'Boomer',
    'boomer@lwaysup.org',
    15554567890
  ),
  (
    2022,
    2,
    'frontend',
    'Franky',
    'frank@lwaysup.org',
    15555678901,
    'Fin',
    'fin@alwaysup.org',
    15556789012
  ),
  (
    2022,
    2,
    'backend',
    'Bonny',
    'bonny@alwaysup.org',
    15557890123,
    'Bea',
    'bea@alwaysup.org',
    15558901234
  );

select * from oncall;

create or replace TEMPORARY view oncall_unpivot as
select * from oncall unpivot exclude nulls 
((name, email, phone) for classification in ((name1, email1, phone1) as primary,
( name2, email2, phone2) as secondary));

select * from oncall_unpivot ;

  

In [0]:
use catalog lakehouse_dev;
use schema datasphere_test;

CREATE OR REPLACE TEMPORARY VIEW sales_by_region(year, quarter, region, sales) AS
   VALUES (2018, 1, 'east', 100),
          (2018, 2, 'east',  20),
          (2018, 3, 'east',  40),
          (2018, 4, 'east',  40),
          (2019, 1, 'east', 120),
          (2019, 2, 'east', 110),
          (2019, 3, 'east',  80),
          (2019, 4, 'east',  60),
          (2018, 1, 'west', 105),
          (2018, 2, 'west',  25),
          (2018, 3, 'west',  45),
          (2018, 4, 'west',  45),
          (2019, 1, 'west', 125),
          (2019, 2, 'west', 115),
          (2019, 3, 'west',  85),
          (2019, 4, 'west',  65);


  select * from sales_by_region;

  create or replace temporary view sales_pivot as
  select * from sales_by_region pivot (sum(sales) for quarter in ( 1 as Q1, 2 as Q2, 3 as Q3, 4 as Q4)) ;

  select * from sales_pivot;


  create or replace temporary view sales_pivot_2 as
  select * from sales_by_region pivot (sum(sales) for (quarter,region) in ( (1,'east') as Q1_East, (1,'west') as Q1_West, (2,'east') as Q2_East, (2,'west') as Q2_West, (3,'east') as Q3_East, (3, 'west') as Q3_West, (4, 'east') as Q4_East, (4, 'west') as Q4_West));

  select * from sales_pivot_2;


SELECT year, q1, q2, q3, q4
  FROM (SELECT year, quarter, sales FROM sales_by_region) AS s
  PIVOT (sum(sales) AS sales
    FOR quarter
    IN (1 AS q1, 2 AS q2, 3 AS q3, 4 AS q4));
 


In [0]:
--SELECT * FROM json.`/Volumes/lakehouse_dev/datasphere_test/managedvolume/`;

create or replace temp view titanic_passengers as select * from json.`/Volumes/lakehouse_dev/datasphere_test/managedvolume/`;

In [0]:
-- download file and put into volumes

-- get file from url and put into managed volume
%python
spark.sql("USE CATALOG lakehouse_dev")
spark.sql("USE SCHEMA datasphere_test")
display(spark.sql("SHOW VOLUMES"))

import requests
import os

response = requests.get("https://health.data.ny.gov/api/views/jxy9-yhdk/rows.csv")
csvfile = response.content.decode('utf-8')

dbutils.fs.put(f"/Volumes/lakehouse_dev/datasphere_test/managedvolume/babynames.csv",csvfile,True)